# LELA60331 Week 5 Seminar: Logistic regression with text

This sheet uses some Python from the **Week 3 Python workbook (LELA60341)**: functions, `enumerate` and `zip`, comprehensions, set operations and `collections.Counter`.

### Matrix-vector multiplication

In our model building so far we have been generating our predicted values by taking the dot product of vectors of weights and vectors of features (and adding the bias). However we want to move on to using more features than just 2. When we have lots of features, it is more efficient to handle the sets of j features for k items as a k x j matrix rather than as j vectors of length k. To combine our weights and our features we then need to take the dot product of each row of our feature matrix with our vector of weights. As you saw in this week's lecture we can take the dot product between a matrix with j columns and a vector of length j. This can be done in numpy as follows:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
vector = np.random.rand(3, 1)
matrix = np.random.rand(10, 3)

In [ ]:
matrix.dot(vector) # or matrix@vector

Remember that in matrix multiplication order is important. The matrix here cannot be left multiplied by the vector because its number of rows does not equal the number of elements in the vector. And so the following doesn't work:

In [ ]:
# This gives a ValueError, because the shapes don't match
vector.dot(matrix)

### Logistic regression with text: sentiment analysis

The dataset we are going to use here is 10000 reviews on Yelp classified as negative (1 or 2 star) or positive (3 or 4 star). We are going to train a classifier using a part of this data and test its performance on another part.

First we download the dataset:

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60331_24-25/refs/heads/main/data/yelp_reviews.txt

The dataset is in a tab-separated file: each line holds a review, then a tab, then the sentiment label. The first line is a header.

We read it into two lists (one for reviews; one for labels) as follows:

In [ ]:
!head -n2 yelp_reviews.txt

In [ ]:
reviews = []
labels = []

with open("yelp_reviews.txt") as fp:
    lines = fp.readlines()
for line in lines[1:]:                         # skip the header line
    fields = line.rstrip().split('\t')        # [review, label]
    reviews.append(fields[0])
    labels.append(fields[1])

print(len(reviews), "reviews;", labels.count("positive"), "positive and", labels.count("negative"), "negative")

We are going to represent our data using one-hot encoding. We need to use the same vocabulary for our training and test data so we do this prior to splitting the data.

In order to one-hot encode we need a list of the vocabulary items to include. We will use the 5000 most frequent words (not an ideal solution but a convenient one). To get this list we split every review into words, count how often each word occurs using a `Counter` from the `collections` module, and take the 5000 most common.

In [ ]:
from collections import Counter

# Tokenise each review: a very naive tokenisation that just splits the text at spaces.
# This gives a list of lists of tokens, one list per review.
tokenized_sents = [review.split() for review in reviews]

# Collapse all the tokens into a single list (extend adds every element of a list to the end)
tokens = []
for sent in tokenized_sents:
    tokens.extend(sent)

# Count the tokens, then keep the 5000 most common words
counts = Counter(tokens)
type_list = [word for word, count in counts.most_common(5000)]

Now we split our data. We are going to use 20% of our data as test items, so we randomly select 8000 indices between 0 and 10000, which are the indices of our training items. The remaining 2000 indices are the indices of our test items: we find them with a **set difference** (all the indices, minus the training ones).

In a real development task we would want to split data into training, development and test. Here we just use training and test.

In [ ]:
np.random.seed(1)        # so that everyone gets the same split
train_ints = np.random.choice(len(reviews), int(len(reviews) * 0.8), replace=False)
test_ints = list(set(range(len(reviews))) - set(train_ints))

labels_train = [labels[i] for i in train_ints]
labels_test = [labels[i] for i in test_ints]

We are now ready to one-hot encode our reviews. With a vocabulary of 5000 words, we want an 8000 x 5000 matrix **M_train** for the training reviews and a 2000 x 5000 matrix **M_test** for the test reviews. In each, row $i$ is a review, column $j$ is a word from the vocabulary, and element $x_{i,j}$ is 1 if word $j$ occurs in review $i$ and 0 otherwise. (Building the two matrices directly, rather than one big matrix that we then split, saves a lot of memory.)

**Activity 1**

We need to build two matrices in the same way, so we write a **function** for it (Week 3 Python workbook). `one_hot(token_lists)` takes a list of tokenised reviews and returns their matrix. The outer loop goes through the reviews with `enumerate`, so `i` is the row. For each word in the review, fill in:

1. the condition: only words that are in our vocabulary have a column (`word_to_index` contains exactly those words);
2. the column for that word.

In [ ]:
# Which column belongs to each word
word_to_index = {word: i for i, word in enumerate(type_list)}

def one_hot(token_lists):
    M = np.zeros((len(token_lists), len(type_list)))
    for i, tokens in enumerate(token_lists):
        for word in tokens:
            if ???:
                M[i, ???] = 1
    return M

M_train = one_hot([tokenized_sents[i] for i in train_ints])
M_test = one_hot([tokenized_sents[i] for i in test_ints])
print(M_train.shape, M_test.shape)

Now we are ready to train our model using the training data.

**Activity 2**

Complete the code below so that it learns a logistic regression model from the training data. It works just like the two-feature logistic regression from last week, but with 5000 features. Instead of a separate line for each weight, we use matrix-vector multiplication (see the top of this sheet):

* `M_train.dot(weights)` gives z (before adding the bias) for every training review at once;
* `M_train.T.dot(q - y)` gives the gradient for every weight at once (before dividing by the number of samples).

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

num_features = 5000
y = np.array([int(label == "positive") for label in labels_train])     # 1 for positive, 0 for negative
weights = np.random.rand(num_features)
bias = np.random.rand(1)
n_iters = 500
lr = 0.4
logistic_loss = []
num_samples = len(y)
eps = 0.00001          # a tiny number, so that we never take the log of 0
for i in range(n_iters):
    z = ???
    q = ???
    loss = -sum(y * np.log2(q + eps) + (1 - y) * np.log2(1 - q + eps))
    logistic_loss.append(loss)

    dw = ???
    db = ???
    weights = ???
    bias = ???

plt.plot(range(1, n_iters), logistic_loss[1:])
plt.xlabel("number of epochs")
plt.ylabel("loss")
plt.show()

Now that we have a fitted model, we can use it to predict labels for our test items. The test reviews are in the one-hot matrix `M_test`. The labels for the test reviews are in the list `labels_test`.

**Activity 3**

Complete the code below so that it calculates the list of predicted values `y_test_pred` for our test items (1 for positive, 0 for negative).

In [ ]:
z = ???
q = ???
y_test_pred = [int(prob > 0.5) for prob in q]

We can calculate accuracy for the performance of our model on the test items as follows:

In [ ]:
y_test = [int(label == "positive") for label in labels_test]
correct = [int(pred == true) for pred, true in zip(y_test_pred, y_test)]
print("accuracy:", sum(correct) / len(correct))

Accuracy is not always a good measure, though, and so we also calculate precision and recall.

**Activity 4**

Calculate precision and recall for the performance of our model on the test data.

1. Complete `confusion_counts`, which counts the true positives, false positives and false negatives. The line for true positives is given: it uses `zip` to walk through the predicted and actual labels side by side.
2. Complete `precision` and `recall`.

Because these are functions, you can reuse them in Activity 5 without copying any code.

In [ ]:
labels_test_pred = ["positive" if pred == 1 else "negative" for pred in y_test_pred]

In [ ]:
def confusion_counts(predicted, actual):
    true_positives = sum([int(p == "positive" and a == "positive") for p, a in zip(predicted, actual)])
    false_positives = ???
    false_negatives = ???
    return true_positives, false_positives, false_negatives

def precision(tp, fp):
    return ???

def recall(tp, fn):
    return ???

tp, fp, fn = confusion_counts(labels_test_pred, labels_test)
print("precision:", precision(tp, fp))
print("recall:", recall(tp, fn))

**Activity 5**

Calculate precision and recall for the performance of the model on the **training** data. You first need predicted labels for the training reviews: do the same as in Activity 3 with `M_train`, then use your functions from Activity 4. How do the numbers differ from those you found for the test set? Why do you think this is?

In [ ]:
q_train = ???
labels_train_pred = ["positive" if prob > 0.5 else "negative" for prob in q_train]
tp, fp, fn = ???
print("precision:", precision(tp, fp))
print("recall:", recall(tp, fn))

### Inspecting weights
Weights in a one-layer network with one-hot encoded inputs are simple to interpret: words with high weights are those whose presence makes a positive outcome most likely, and those with low weights are those whose presence makes a negative outcome most likely.

Our weights are in a numpy array, with each weight corresponding to the word at the same position (index) in our `type_list`.

We can obtain a list of the indices in order of the weight at each position using the numpy function `argsort`. This sorts from low to high, so we take the first 20 to find the indices of the twenty words most strongly associated with a negative outcome. Or we can reverse it to find the indices of the twenty words most strongly associated with a positive outcome.

We can then just look at the words that occur at these indices:

In [ ]:
[type_list[i] for i in np.argsort(weights)[:20]]

In [ ]:
[type_list[i] for i in np.argsort(weights)[::-1][:20]]

**Activity 6**

Inspect the list of tokens with highest and lowest weights, going beyond the top and bottom 20. Do you spot any that indicate overfitting?

**Activity 7**

Look back up to the cell where we tokenise the reviews (`tokenized_sents = ...`). Can you change the way the text is tokenised in order to avoid the problematic tokens you noted in Activity 6? (The string methods from Week 2 and the comprehensions from Week 3 of the Python workbook will help.) Retrain the model. How do your changes affect performance and the token lists?